# 02 · Build the alignment matrix

Runs the full pipeline (or loads a previous run), then explores the country × competency alignment interactively: raw heatmap, per-competency ranking of countries, and a clustered heatmap.

If you have no strategy texts yet, run `python -m scripts.make_sample_data` from the repo root first.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src import alignment, competencies, config, visualize
from src.models import load_embedding_model

comp_df = competencies.load_competencies()
chunk_frame = alignment.build_chunk_frame()
print('countries:', chunk_frame.country.nunique(), '| chunks:', len(chunk_frame))
print('skipped (no file):', chunk_frame.attrs.get('skipped'))

In [ ]:
model = load_embedding_model()
corpus = competencies.embedding_texts(comp_df) + chunk_frame.chunk_text.tolist()
model.fit(corpus)
comp_embeds = competencies.embed_competencies(comp_df, model, save_path=None)
scores = alignment.build_alignment_scores(chunk_frame, comp_df, comp_embeds, model,
                                          cache_embeddings=False)
pivot = visualize.to_pivot(scores, competency_order=comp_df.competency_id.tolist())
pivot.round(3)

In [ ]:
# Raw heatmap (renders inline; pass save_path=... to also write a PNG)
visualize.plot_heatmap(pivot)
import matplotlib.pyplot as plt; plt.show()

In [ ]:
# Which countries emphasise each competency most?
for cid in comp_df.competency_id:
    top = pivot[cid].sort_values(ascending=False).head(3)
    name = comp_df.loc[comp_df.competency_id == cid, 'name'].iloc[0]
    print(f"{cid} {name}:", ', '.join(f'{c} ({v:.2f})' for c, v in top.items()))

In [ ]:
# Clustered heatmap groups similar countries and similar competencies together
visualize.plot_clustered_heatmap(pivot, normalise='global')
import matplotlib.pyplot as plt; plt.show()